# Story maps from the Python API

A story map walks a reader through a map one stop at a time. Each stop is a *story segment*: a map view (a center and a zoom level) plus the text, image and layer settings to use while that stop is on screen.

This notebook builds a short story about France entirely from Python, then you present it from the map toolbar.

The document is saved as `story-python-api.jGIS`. Saving it to a file matters here: the map widget only gets a toolbar when it is backed by a file, and the story editor and the presentation preview live in that toolbar.

In [ ]:
from jupytergis import GISDocument

doc = GISDocument("story-python-api.jGIS", latitude=46.6, longitude=2.5, zoom=5)
await doc.ready()

doc

## Start from a clean slate

A document holds a single story, and `add_story_segment` always appends. Re-running this notebook over the same file would therefore stack a second tour on top of the first, and `create_story` would raise because a story is already there.

Clearing the layers first makes the notebook safe to run again. Removing a segment also drops it from the story, so this empties the story as well.

In [ ]:
for layer_id in list(doc.layers):
    doc.remove_layer(layer_id)

## Add the layers first

A story segment only moves the camera and switches content. It carries no data of its own, so add the layers you want the reader to see before you build the story.

In [ ]:
doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap",
)

regions = doc.add_geojson_layer(
    name="French regions",
    path="data/france_regions.geojson",
    opacity=0.3,
)

## Create the story

`story_type` is either `"guided"`, where the reader steps from segment to segment, or `"Vertical Scroll"`, where the reader scrolls through them.

`create_story` is for a document with no story yet, and `update_story` changes the one that is already there. The story left behind by an earlier run of this notebook survives the clearing above, so the cell below handles both cases.

You can skip `create_story` entirely: the first `add_story_segment` creates a story named "New Story" with the default settings.

In [ ]:
story = dict(
    title="A short tour of France",
    story_type="guided",
    presentation_bg_color="#0f1b24",
    presentation_text_color="#f4f1ea",
    story_panel_opacity=0.9,
)

if doc.story is None:
    doc.create_story(**story)
else:
    doc.update_story(**story)

## What a segment can be given

`add_story_segment` takes one parameter per thing a stop can control, and every one of them is optional. The docstring is the quickest reference:

In [ ]:
help(doc.add_story_segment)

The three that shape a stop the most:

- **Where**: `center` is a `(longitude, latitude)` pair, `zoom` the map zoom level. Pass a `[west, south, east, north]` box as `extent` when that is what you have, and the map flies to the middle of it.
- **What the reader sees**: `content_mode="map"` keeps the map on screen and puts the text in a panel over it, placed by `pane_alignment` and `panel_width`. `content_mode="markdown"` hides the map and gives the text the whole screen. Passing `markdown` without a `content_mode` picks `"markdown"` for you, so ask for `"map"` when you want text over a visible map.
- **How it gets there**: `transition` is `"linear"` to fly straight, `"smooth"` to zoom out and back in, or `"immediate"` to jump. `transition_time` is in seconds, and the story editor allows up to 3.

## The opening card

The story opens on a title with no map behind it, so leave `content_mode` at its default.

In [ ]:
doc.add_story_segment(
    name="Title",
    center=(2.5, 46.6),
    zoom=5,
    markdown=(
        "# A short tour of France\n\n"
        "Three stops, from the whole country down to a single harbour."
    ),
    transition="immediate",
)

## Stop one: the country

The map comes back here, with the text in a narrow panel on the left so the country stays clear.

In [ ]:
doc.add_story_segment(
    name="The country",
    center=(2.5, 46.6),
    zoom=5,
    markdown=(
        "## The country\n\n"
        "France is a little over 550,000 square kilometres, "
        "and the thirteen outlined shapes are its mainland regions."
    ),
    content_mode="map",
    pane_alignment="start",
    panel_width="28%",
    transition="linear",
    transition_time=1.5,
)

## Stop two: Paris

A tighter view, the same panel placement, and the identify control switched on so the reader can click a region and read its name.

In [ ]:
doc.add_story_segment(
    name="Paris",
    center=(2.358213, 48.853757),
    zoom=11,
    markdown=(
        "## Paris\n\n"
        "The Seine runs right through the middle of it, "
        "and the city sits inside the smallest region of the thirteen."
    ),
    content_mode="map",
    pane_alignment="start",
    panel_width="28%",
    transition="linear",
    transition_time=1.5,
    enable_identify=True,
)

## Stop three: Marseille

A bounding box instead of a center, and a photograph in the panel.

In [ ]:
doc.add_story_segment(
    name="Marseille",
    extent=[5.2, 43.2, 5.5, 43.4],
    zoom=12,
    markdown=(
        "## Marseille\n\n"
        "Some six hundred and sixty kilometres south of Paris, on the Mediterranean, "
        "and the oldest city in the country."
    ),
    image="https://upload.wikimedia.org/wikipedia/commons/thumb/1/1a/Vieux-Port_de_Marseille.jpg/960px-Vieux-Port_de_Marseille.jpg",
    image_caption="The old port",
    content_mode="map",
    pane_alignment="start",
    panel_width="28%",
    transition="smooth",
    transition_time=2,
)

## The closing stop, with the outlines brought forward

`layer_overrides` changes how the *other* layers are drawn for as long as a segment is on screen, and puts them back afterwards. `targetLayer` is the id returned when the layer was added.

The regions went in at 0.3 opacity. They come up to full strength for this last stop.

In [ ]:
doc.add_story_segment(
    name="The regions",
    center=(2.5, 46.6),
    zoom=5,
    markdown=(
        "## Back out\n\nThe outlines come forward, and the tour ends where it started."
    ),
    content_mode="map",
    pane_alignment="start",
    panel_width="28%",
    transition="smooth",
    transition_time=2,
    layer_overrides=[
        {"targetLayer": regions, "visible": True, "opacity": 1.0},
    ],
)

## Read the story back, and change it

`update_story` changes the story properties and leaves alone whatever you do not pass. `doc.story` gives the story properties and `doc.story_segments` the segment ids in presentation order.

In [ ]:
doc.update_story(story_panel_opacity=0.85)

print(doc.story["title"])
[doc.layers[segment]["name"] for segment in doc.story_segments]

## Remove a segment

A segment is a layer, so `remove_layer` removes it and drops it from the story at the same time.

In [ ]:
scratch = doc.add_story_segment(name="Scratch", center=(2.5, 46.6), zoom=5)
doc.remove_layer(scratch)

[doc.layers[segment]["name"] for segment in doc.story_segments]

## Present it

Use the toolbar of the widget above, or open `story-python-api.jGIS` from the file browser, and click the story button. That opens the story editor, where you can preview the presentation and keep editing the segments by hand.

## What the Python API does not do yet

- A segment can only be appended. There is no `update_story_segment` and no way to reorder segments from Python, so edits to an existing segment go through the story editor.
- There is no `remove_story`. Removing every segment leaves an empty story behind, which is why this notebook updates the story instead of creating it when it is run again.
- Markdown attachments, meaning images pasted into a segment and stored inside the document, are not exposed.
- Presenting is a user interface action. Nothing in Python starts or steps through the presentation.
- QGIS documents (`.qgs` and `.qgz`) cannot hold a story, so these calls raise on them.